# Biblical Qwen 2.5 14B Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** Qwen 2.5 14B Instruct

**Dataset:** Augmentoolkit-generated Biblical persona (first-person apostolic responses)

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Deployment Target:** A5000 GPU server via vLLM (24GB VRAM — 14B 4-bit ~8GB fits comfortably)

**Chat Template:** `<|im_start|>role\ncontent<|im_end|>` (ChatML)

## Step 1: Configuration

All paths and variables for easy configuration.

In [ ]:
# ============================================================================
# CONFIGURATION - All variables for easy setup
# ============================================================================
# Training: DGX Spark (128GB unified memory)
# Deployment: A5000 GPU server (24GB VRAM) via vLLM (14B 4-bit ~8GB)

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM = "unsloth/Qwen2.5-14B-Instruct-bnb-4bit"
MODEL_NAME_BASE = "biblical_qwen25_14b_instruct_unsloth_4bit_nemotron_safety_guard_lora"

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR = f"./output/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
TARGET_EPOCHS = 1

# =========================== LoRA CONFIGURATION ===========================
# r=8, alpha=16: conservative, preserves base model behavior
# Increase r to 16-64 for more aggressive adaptation
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0

# Target modules:
#   Conservative (persona): ["q_proj", "v_proj"]
#   Balanced:               ["q_proj", "k_proj", "v_proj", "o_proj"]
#   Aggressive (knowledge): ["q_proj", "k_proj", "v_proj", "o_proj",
#                            "gate_proj", "up_proj", "down_proj"]
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# =========================== VLLM DEPLOYMENT ===========================
VLLM_MODELS_DIR = "/home/spark/projects/stoic/output/vllm"

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "I am struggling with forgiveness. What does Scripture teach about forgiving others?"

# ============================================================================
print("\u2713 Configuration loaded (14B 4-bit QLoRA version)")
print(f"  Base model: {BASE_LLM}")
print(f"  Model name: {MODEL_NAME_BASE}")
print(f"  Input data: {INPUT_DATA_PATH}")
print(f"  Output base: {OUTPUT_BASE_DIR}")
print(f"  LoRA config: r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"  Training: batch={BATCH_SIZE}, grad_accum={GRAD_ACCUM}, lr={LEARNING_RATE}")
print(f"  Training precision: 4-bit QLoRA")
print(f"  Deployment: vLLM on A5000 GPU server")

✓ Configuration loaded (14B 4-bit QLoRA version)
  Base model: unsloth/Qwen2.5-14B-Instruct-bnb-4bit
  Model name: biblical_qwen25_14b_instruct_unsloth_4bit
  Input data: /home/spark/projects/augmentoolkit/outputs/biblical_persona_dataset
  Output base: ./output/biblical_qwen25_14b_instruct_unsloth_4bit
  LoRA config: r=8, alpha=16, targets=['q_proj', 'v_proj']
  Training: batch=1, grad_accum=8, lr=0.0001
  Training precision: 4-bit QLoRA
  Deployment: vLLM on A5000 GPU server


## 2. Environment Preparation

Install Unsloth and updated HuggingFace libraries.

In [2]:
# Install core packages from PyPI (much faster than git installs)
!pip install -q unsloth transformers trl peft accelerate datasets bitsandbytes

# Verify installations
import unsloth
import transformers
import trl
print(f"\u2713 Unsloth: {unsloth.__version__}")
print(f"\u2713 Transformers: {transformers.__version__}")
print(f"\u2713 TRL: {trl.__version__}")
print("Environment ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  queued_call()


🦥 Unsloth Zoo will now patch everything to make training faster!
✓ Unsloth: 2026.2.1
✓ Transformers: 4.57.3
✓ TRL: 0.24.0
Environment ready!


## 3. Load Dataset & Format for Instruction Tuning

Load the Augmentoolkit-generated Biblical dataset (first-person apostolic responses from Paul, David, Peter, etc.).

In [ ]:
from datasets import load_dataset

# === SAFETY GUARD DATASET ===
print("Loading Nemotron Safety Guard Dataset v3...")

dataset = load_dataset(
    "nvidia/Nemotron-Safety-Guard-Dataset-v3", 
    split="train"   # or "all" if you want everything
)

# Optional: Filter to only unsafe prompts with good refusal responses (recommended for guard LoRA)
dataset = dataset.filter(lambda x: x["prompt_label"] == "unsafe" and x["response"] is not None)

print(f"Loaded {len(dataset)} safety examples")

# Format for your existing format_instruct function (ChatML)
def format_safety(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_safety, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: len(x["text"]) > 50)  # remove garbage

print(f"Final safety dataset: {len(dataset)} examples ready for training")

LOADING ALL AUGMENTOOLKIT DATA
Found 50 subdirectories
  Loaded 123 from factual_sft_full_biblical_data_followup_0/simplified_data_rag.jsonl
  Loaded 123 from factual_sft_full_biblical_data_followup_0/plain_qa_list.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_1/simplified_data_rag.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_1/plain_qa_list.jsonl
  Loaded 121 from factual_sft_full_biblical_data_followup_2/simplified_data_rag.jsonl
  Loaded 121 from factual_sft_full_biblical_data_followup_2/plain_qa_list.jsonl
  Loaded 113 from factual_sft_full_biblical_data_followup_3/simplified_data_rag.jsonl
  Loaded 113 from factual_sft_full_biblical_data_followup_3/plain_qa_list.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_4/simplified_data_rag.jsonl
  Loaded 114 from factual_sft_full_biblical_data_followup_4/plain_qa_list.jsonl
  Loaded 112 from factual_sft_full_biblical_data_followup_5/simplified_data_rag.jsonl
  Loaded 112 from factual_sft

Generating train split: 0 examples [00:00, ? examples/s]

  Skipped Augmentoolkit-Augmentoolkit-Pippa-Thoughts_0.jsonl: An error occurred while generating the dataset



Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Bluemoon-1mil-thoughts_0.jsonl: An error occurred while generating the dataset


...

  Skipped Augmentoolkit-Augmentoolkit-LMsys-800k-Thoughts_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Openthoughts-100mil-DifferentFormat_0.jsonl: An error occurred while generating the dataset


Generating train split: 0 examples [00:00, ? examples/s]


  Skipped Augmentoolkit-Augmentoolkit-Capybara-2point5mil-Thoughts_0.jsonl: An error occurred while generating the dataset
  Loaded 2309 from inferred_facts_full_biblical_data/final_output.jsonl
  Loaded 26 from pretraining_run/text_chunks_full_biblical_data.jsonl
  Loaded 2199 from pretraining_run/representation_variation_full_biblical_data.jsonl
  Loaded 2309 from pretraining_run/inferred_facts_full_biblical_data.jsonl
  Loaded 134 from rag_data_full_biblical_data/axolotl_rag_conversations.jsonl
  Loaded 5932 from rag_source_data/rag_data_full_biblical_data.jsonl
  Loaded 5932 from rag_source_data/rag_data_combined.jsonl
  Loaded 2199 from representation_variation_full_biblical_data/final_output.jsonl
  Loaded 26 from sft_run/pretraining_subset_441912.jsonl
  Loaded 134 from sft_run/axolotl_rag_conversations_full_biblical_data.jsonl

Total dataset loaded: 30274 examples
Columns: ['conversations', 'text', 'metadata', 'segments', 'question', 'source_text', 'source_metadata', 'related_c

## 4. Load Model & Tokenizer with Unsloth (4-bit)

Load Qwen 2.5 14B Instruct model in **4-bit precision** for QLoRA training.

**Training on DGX Spark (128GB unified memory):**
- 14B in 4-bit fits comfortably (~8GB)
- Plenty of headroom for optimizer states and activations

**Deployment on A5000 (24GB VRAM):**
- 14B 4-bit ~8GB, fits within 24GB with room for KV-cache
- vLLM handles KV-cache management efficiently

In [4]:
from unsloth import FastLanguageModel
import torch

# Load model in 4-bit precision for QLoRA training
# Use device_map={"":  0} to force all layers onto GPU 0
# DGX Spark has 119.7GB \u2014 plenty for 14B 4-bit (~8GB)
# device_map="auto" incorrectly offloads some layers to CPU which BnB 4-bit rejects
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,           # Auto-detect (will use bfloat16 for compute)
    load_in_4bit=True,    # 4-bit quantization for QLoRA
    device_map={"":  0}    # Force all layers onto GPU 0 (119GB unified memory)
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"\u2713 Model loaded: {BASE_LLM}")
print(f"\u2713 Precision: 4-bit (QLoRA)")
print(f"\u2713 Tokenizer configured")
print(f"\u2713 Max sequence length: {MAX_SEQ_LENGTH}")

==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 2/2 [00:51<00:00, 25.96s/it]


✓ Model loaded: unsloth/Qwen2.5-14B-Instruct-bnb-4bit
✓ Precision: 4-bit (QLoRA)
✓ Tokenizer configured
✓ Max sequence length: 2048


In [5]:
# Format dataset for Qwen 2.5 chat template (ChatML)
# Handle BOTH ShareGPT format (conversations) AND raw text format
def format_instruct(example):
    # If has conversations field AND it's not null, convert ShareGPT to chat template
    if example.get("conversations") is not None:
        messages = []
        for turn in example["conversations"]:
            # ShareGPT uses "from": "system"/"human"/"gpt"
            # Standard uses "role": "system"/"user"/"assistant"
            if turn["from"] == "system":
                messages.append({"role": "system", "content": turn["value"]})
            elif turn["from"] == "human":
                messages.append({"role": "user", "content": turn["value"]})
            elif turn["from"] == "gpt":
                messages.append({"role": "assistant", "content": turn["value"]})
        
        text = tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        return {"text": text}
    
    # Otherwise if it has a text field (raw text files), keep it as-is
    elif example.get("text") is not None and len(str(example["text"])) > 0:
        return {"text": str(example["text"])}
    
    # Skip malformed examples
    return {"text": ""}

# Format and keep only text column
dataset = dataset.map(format_instruct, remove_columns=dataset.column_names)

# Filter out empty examples
dataset = dataset.filter(lambda x: len(x["text"]) > 0)

print(f"\n--- Sample formatted text (first 500 chars) ---")
print(dataset[0]['text'][:500])
print(f"\n\u2713 Dataset formatted: {len(dataset)} examples")

Filter: 100%|██████████| 30274/30274 [00:00<00:00, 877328.32 examples/s]


--- Sample formatted text (first 500 chars) ---
[[[OVERALL_CONTEXT_IS -> Biblical Faith and Apostolic Teaching]]]
Specific source: habakkuk.txt
The ultimate consequence of Habakkuk ' s vision is that the Lord God is his strength, andhe will make HahaukKk 's feet likehinds ' fSst, WnaGlinN him to walk upon his high places. This is the culmination of a series of events that began with Habakkuk ' scryto God, feeling that God was not hearing him. However, God had already set in motion a plan to worka work in the days of Habakkuk ' s audience that

✓ Dataset formatted: 18142 examples


## 5. Add LoRA Adapters

Configure LoRA for efficient fine-tuning. See configuration section for module explanations.

In [6]:
from peft import LoraConfig

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH
)

print(f"\u2713 LoRA adapters added (r={LORA_R}, targets={LORA_TARGET_MODULES})")
print(f"\u2713 Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Not an error, but Unsloth cannot patch O projection layer with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.2.1 patched 48 layers with 48 QKV layers, 0 O layers and 0 MLP layers.


✓ LoRA adapters added (r=8, targets=['q_proj', 'v_proj'])
✓ Trainable parameters: 6,291,456


## 6. Trainer Setup & Training

**PURE DOMAIN DATA with LIGHT TRAINING:**
- 100% authentic Biblical examples from epistles and psalms
- 1 epoch with low learning rate (1e-4) to gently teach persona
- 14B is already highly capable \u2014 minimal fine-tuning is sufficient
- This preserves base model capabilities while adding authentic voice

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Calculate training steps
effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(dataset) // effective_batch_size
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = max(1, max_steps // 10)
save_steps = min(100, steps_per_epoch)  # Save every 100 steps or at epoch end

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        report_to="none",
        save_strategy="steps",
        save_steps=save_steps,
    )
)

print("\u2713 Trainer configured for 14B 4-bit QLoRA training")
print(f"\u2713 Dataset size: {len(dataset)} conversations")
print(f"\u2713 Effective batch size: {effective_batch_size} (batch={BATCH_SIZE} \u00d7 grad_accum={GRAD_ACCUM})")
print(f"\u2713 Steps per epoch: {steps_per_epoch}")
print(f"\u2713 Total epochs: {TARGET_EPOCHS}")
print(f"\u2713 Total steps: {max_steps}")
print(f"\u2713 Warmup steps: {warmup_steps}")
print(f"\u2713 Save every: {save_steps} steps")
print(f"\u2713 Learning rate: {LEARNING_RATE}")

Unsloth: Tokenizing ["text"] (num_proc=24): 100%|██████████| 18142/18142 [00:03<00:00, 4607.43 examples/s]

✓ Trainer configured for 14B 4-bit QLoRA training
✓ Dataset size: 18142 conversations
✓ Effective batch size: 8 (batch=1 × grad_accum=8)
✓ Steps per epoch: 2267
✓ Total epochs: 1
✓ Total steps: 2267
✓ Warmup steps: 226
✓ Save every: 100 steps
✓ Learning rate: 0.0001


In [8]:
# Start training
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18,142 | Num Epochs = 1 | Total steps = 2,267
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 6,291,456 of 14,776,325,120 (0.04% trained)


Step,Training Loss
5,1.398700
10,1.621700
15,1.821400
20,1.640600
25,1.654800
30,1.729700
35,1.671600
40,1.540600
45,1.623500
50,1.568700


TrainOutput(global_step=2267, training_loss=0.8608607439727682, metrics={'train_runtime': 17976.3436, 'train_samples_per_second': 1.009, 'train_steps_per_second': 0.126, 'total_flos': 8.519883162129715e+17, 'train_loss': 0.8608607439727682, 'epoch': 0.9996692757138133})

## 7. Save LoRA Adapters

Save the trained LoRA adapters separately. These can be loaded on any Qwen 2.5 14B model later.

**This is the primary output** - merging to full model is optional (Step 8).

In [ ]:
from pathlib import Path

# Save LoRA adapters (primary output)
LORA_OUTPUT_DIR = f"{OUTPUT_BASE_DIR}/lora_adapters"
Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

print(f"\u2713 LoRA adapters saved!")
print(f"   Path: {LORA_OUTPUT_DIR}")
print(f"   Can be loaded on any Qwen 2.5 14B model with PEFT")

[tornado.application|ERROR]Exception in callback functools.partial(<bound method OutStream._flush of <ipykernel.iostream.OutStream object at 0xe12e2faf3c10>>)
Traceback (most recent call last):
  File "/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 103, in json_packer
    ).encode("utf8", errors="surrogateescape")
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'utf-8' codec can't encode characters in position 28-29: surrogates not allowed

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/tornado/ioloop.py", line 758, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/home/spark/projects/quantization/.venv/lib/python3.12/site-packages/ipykernel/iostream.py", line 644, in _flush
    self.session.send(
  File "/home/spark/projects/quantization/.venv/lib/python3.12/site

✅ LoRA adapters saved!
   Path: ./output/biblical_qwen25_14b_instruct_unsloth_4bit/lora_adapters
   Can be loaded on any Qwen 2.5 14B model with PEFT
